# Day 5 · Exercise 5: The Hardened safe_ask

**What you'll build:** `safe_ask(question: str, fallback: str = "") -> str`
— an Ollama call with logging at entry and exit, and error handling that
returns a fallback string when Ollama is unavailable.

**Why it matters:** This is the production-ready pattern for any LLM-calling
function. Every function you write for the rest of this course that calls an
LLM should have these three layers: logging, error handling, and a clear
fallback strategy.

> **Ollama required** — make sure it's running before you start.

## Setup (already defined)

In [ ]:
import logging
import ollama

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)

logger = logging.getLogger(__name__)

MODEL = "llama3.2"
SYSTEM = "You are a helpful assistant. Answer concisely."

## Your Implementation

In [ ]:
def safe_ask(question: str, fallback: str = "") -> str:
    """Ask the local LLM. Returns fallback string if Ollama is unavailable.

    Steps:
    1. Log at DEBUG: the function was called (include first 60 chars of question)
    2. try:
           Call ollama.chat(model=MODEL, messages=[system + user turns])
           Extract the answer from response["message"]["content"]
           Log at INFO: how many chars you got back
           Return the answer
    3. except Exception as e:
           Log at ERROR: what failed
           If fallback is truthy, return fallback
           Otherwise, raise (re-raise the exception)

    Args:
        question: The question to ask the model.
        fallback: Return this string if the LLM call fails.
                  If empty and the call fails, the exception propagates.

    Returns:
        The model's answer, or fallback if the call fails.

    Raises:
        Exception: If fallback is empty and the LLM call fails.

    Example:
        answer = safe_ask("What is 2+2?", fallback="I don't know.")
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists with fallback parameter
    try:
        assert callable(safe_ask), 'safe_ask is not defined'
        import inspect
        sig = inspect.signature(safe_ask)
        assert 'fallback' in sig.parameters, 'function should accept fallback parameter'
        print(f'{_PASS} Check 1/{total}: safe_ask exists and accepts fallback parameter')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama is running
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: run ollama serve')
        return
    print(f'{_PASS} Check 2/{total}: Ollama server is reachable')
    score += 1

    # Check 3: returns a non-empty string for a real question
    try:
        result = safe_ask("Say 'hello' in one word.")
        assert isinstance(result, str) and len(result) > 0, \
            f'expected non-empty str, got {result!r}'
        print(f'{_PASS} Check 3/{total}: safe_ask() returns a non-empty string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: fallback is returned when model is invalid
    global MODEL
    _orig = MODEL
    MODEL = "no-such-model-xyz-day005"
    try:
        result = safe_ask("hello", fallback="fallback-value")
        assert result == "fallback-value", \
            f'expected "fallback-value", got {result!r}'
        print(f'{_PASS} Check 4/{total}: fallback returned when model is invalid')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: exception raised instead of returning fallback — {e}')
    finally:
        MODEL = _orig

    # Check 5: exception propagates when no fallback and model is invalid
    MODEL = "no-such-model-xyz-day005"
    try:
        result = safe_ask("hello")  # no fallback
        print(f'{_FAIL} Check 5/{total}: should have raised an exception, got {result!r}')
    except Exception:
        print(f'{_PASS} Check 5/{total}: exception propagates when no fallback provided')
        score += 1
    finally:
        MODEL = _orig

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 5 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Add a `max_tries` parameter so the function retries on failure before
returning the fallback:

```python
def safe_ask(question: str, fallback: str = "", max_tries: int = 1) -> str:
    last_error = None
    for attempt in range(max_tries):
        try:
            logger.debug("safe_ask() | attempt %d", attempt + 1)
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user",   "content": question},
                ],
            )
            return response["message"]["content"]
        except Exception as e:
            last_error = e
            logger.warning("safe_ask() | attempt %d failed: %s", attempt + 1, e)
    logger.error("safe_ask() | all %d attempts failed", max_tries)
    if fallback:
        return fallback
    raise last_error
```

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import logging
import ollama

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)

logger = logging.getLogger(__name__)

def safe_ask(question: str, fallback: str = "") -> str:
    logger.debug("safe_ask() | question=%r", question[:60])
    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user",   "content": question},
            ],
        )
        answer = response["message"]["content"]
        logger.info("safe_ask() | got %d chars", len(answer))
        return answer
    except Exception as e:
        logger.error("safe_ask() failed: %s", e)
        if fallback:
            return fallback
        raise
```

**Key details:** (1) `logger.debug` at entry with the first 60 chars of the question.
(2) `logger.info` on success with the response length. (3) `logger.error` on failure.
(4) Return `fallback` if it's truthy; use bare `raise` if not (preserves the original
exception and traceback). (5) `MODEL` and `SYSTEM` are read from the module-level
globals, so patching `MODEL` in the check cell affects this function.
</details>